In [3]:
!pip install pandas numpy scikit-learn nltk

In [4]:
import pandas as pd
import numpy as np
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
df = pd.read_csv("spam.csv", encoding='latin-1')

# Keep only required columns
df = df[['v1','v2']]

# Rename columns
df.columns = ['label','message']

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [7]:
df['label'] = df['label'].map({'ham':0, 'spam':1})

print(df.head())

   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [8]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):

    text = text.lower()

    text = ''.join([ch for ch in text if ch not in string.punctuation])

    words = text.split()

    words = [stemmer.stem(word) for word in words if word not in stop_words]

    return " ".join(words)

df['clean_message'] = df['message'].apply(preprocess)

print(df[['message','clean_message']].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go jurong point crazi avail bugi n great world...  
1                              ok lar joke wif u oni  
2  free entri 2 wkli comp win fa cup final tkt 21...  
3                u dun say earli hor u c alreadi say  
4          nah dont think goe usf live around though  


In [9]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df['clean_message'])

y = df['label']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
model = MultinomialNB()

model.fit(X_train, y_train)

MultinomialNB()

In [12]:
y_pred = model.predict(X_test)

In [13]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9614349775784753


In [14]:
print("\nClassification Report\n")

print(classification_report(y_test, y_pred))


Classification Report

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.71      0.83       150

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [15]:
print("Confusion Matrix\n")

print(confusion_matrix(y_test, y_pred))

Confusion Matrix

[[965   0]
 [ 43 107]]


In [16]:
email = input("Enter Email Message: ")

email = preprocess(email)

email_vector = vectorizer.transform([email])

prediction = model.predict(email_vector)

if prediction[0] == 1:
    print("Spam Email")
else:
    print("Not Spam")

Enter Email Message: Offer letter
Not Spam


In [17]:
print("\n========== Spam Analysis Report ==========")

print("Total Emails :", len(df))
print("Spam Emails  :", sum(df['label']==1))
print("Ham Emails   :", sum(df['label']==0))

print("\nSpam Percentage : {:.2f}%".format(
    (sum(df['label']==1)/len(df))*100))

print("Ham Percentage  : {:.2f}%".format(
    (sum(df['label']==0)/len(df))*100))

print("\nModel Accuracy : {:.2f}%".format(accuracy*100))


========== Spam Analysis Report ==========
Total Emails : 5572
Spam Emails  : 747
Ham Emails   : 4825

Spam Percentage : 13.41%
Ham Percentage  : 86.59%

Model Accuracy : 96.14%
